In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: Si, SEPD

This example demonstrates a Rietveld refinement of Si crystal
structure using time-of-flight neutron powder diffraction data from
SEPD at Argonne.

It also shows how to switch calculation engine and peak profile type.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [3]:
structure = StructureFactory.from_scratch(name='si')

### Set Space Group

In [4]:
structure.space_group.name_h_m = 'F d -3 m'
structure.space_group.coord_system_code = '2'

### Set Unit Cell

In [5]:
structure.cell.length_a = 5.431

### Set Atom Sites

In [6]:
structure.atom_sites.create(
    id='Si',
    type_symbol='Si',
    fract_x=0.125,
    fract_y=0.125,
    fract_z=0.125,
    adp_iso=0.5,
)

## 🔬 Define Experiment

This section shows how to add experiments, configure their
parameters, and link the structures defined in the previous step.

### Download Data

In [7]:
data_path = download_data('meas-si-sepd', destination='data')

Getting data...


Data 'meas-si-sepd': Si, SEPD (Argonne)


✅ Data 'meas-si-sepd' downloaded to '../../../data/meas-si-sepd.xye'


### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(
    name='sepd',
    data_path=data_path,
    beam_mode='time-of-flight',
)

### Set Instrument

In [9]:
expt.instrument.setup_twotheta_bank = 144.845
expt.instrument.calib_d_to_tof_offset = -10.0
expt.instrument.calib_d_to_tof_linear = 7476.91
expt.instrument.calib_d_to_tof_quadratic = -1.54

### Set Peak Profile

In [10]:
expt.peak.show_supported()

Peak types


,,Type,Description
1,,pseudo-voigt,TOF non-convoluted pseudo-Voigt profile
2,*,jorgensen,TOF Jorgensen profile: back-to-back exponentials ⊗ Gaussian
3,,jorgensen-von-dreele,TOF Jorgensen-Von Dreele profile: back-to-back exponentials ⊗ pseudo-Voigt
4,,double-jorgensen-von-dreele,TOF Double-Jorgensen-Von Dreele profile: double back-to-back exponentials ⊗ pseudo-Voigt (Z-Rietveld type0m)


In [11]:
expt.peak.type = 'jorgensen-von-dreele'

⚠️ Switching peak profile type adds these settings with defaults:
• broad_lorentz_gamma_0=0.0
• broad_lorentz_gamma_1=0.0
• broad_lorentz_gamma_2=0.0
• broad_lorentz_size=0.0
• broad_lorentz_strain=0.0


Peak profile type for experiment 'sepd' changed to


jorgensen-von-dreele


In [12]:
expt.peak.broad_gauss_sigma_0 = 3.0148
expt.peak.broad_gauss_sigma_1 = 33.3451
expt.peak.broad_gauss_sigma_2 = 0.0
expt.peak.broad_lorentz_gamma_0 = 0.0
expt.peak.broad_lorentz_gamma_1 = 2.5489
expt.peak.broad_lorentz_gamma_2 = 0.0
expt.peak.rise_alpha_0 = 0.0
expt.peak.rise_alpha_1 = 0.5971
expt.peak.decay_beta_0 = 0.0408
expt.peak.decay_beta_1 = 0.0123

In [13]:
expt.peak.cutoff_fwhm = 8.2

### Set Background

In [14]:
expt.background.auto_estimate()

### Set Linked Structures

In [15]:
expt.linked_structures.create(structure_id='si', scale=600.0)

## 📦 Define Project

The project object is used to manage the structure, experiment, and
analysis.

### Create Project

In [16]:
project = Project(name='si_sepd')

### Add Structure

In [17]:
project.structures.add(structure)

### Add Experiment

In [18]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section shows the analysis process, including how to set up
calculation and fitting engines.

### Display Structure

In [19]:
project.display.structure(struct_name='si')

Structure 🧩 'si' (Atom view type: 'covalent')


### Display Pattern

In [20]:
project.display.pattern(expt_name='sepd')
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 1/4

Set parameters to be refined.

In [21]:
structure.cell.length_a.free = True

expt.linked_structures['si'].scale.free = True
expt.instrument.calib_d_to_tof_offset.free = True

Show free parameters after selection.

In [22]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43100,,-inf,inf,Å
2,sepd,linked_structure,si,scale,600.00000,,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,-10.00000,,-inf,inf,μs


#### Run Fitting

In [23]:
project.analysis.minimizer.type = 'bumps (lm)'

⚠️ Switching minimizer type removes these settings:
• gradient_tolerance


Current minimizer changed to


bumps (lm)


In [24]:
project.analysis.minimizer.chi_square_change_tolerance = 1e-2

In [25]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),7.61
4,📏 Goodness-of-fit (reduced χ²),6.91
5,"📏 R-factor (Rf, %)",11.59
6,"📏 R-factor squared (Rf², %)",6.19
7,"📏 Weighted R-factor (wR, %)",9.62


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4310,5.4308,0.0001,0.00 % ↓
2,sepd,linked_structure,si,scale,,600.0000,367.9696,0.9626,38.67 % ↓
3,sepd,instrument,,d_to_tof_offset,μs,-10.0000,-8.3525,0.0760,16.48 % ↓


#### Display Pattern

In [26]:
project.display.pattern(expt_name='sepd')

In [27]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 2/4

Set more parameters to be refined.

In [28]:
for point in expt.background:
    point.intensity.free = True

Show free parameters after selection.

In [29]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43084,0.00006,-inf,inf,Å
2,sepd,linked_structure,si,scale,367.96957,0.96260,-inf,inf,
3,sepd,instrument,,d_to_tof_offset,-8.35245,0.07597,-inf,inf,μs
4,sepd,background,1,intensity,213.55062,,-inf,inf,
5,sepd,background,2,intensity,117.61669,,-inf,inf,
6,sepd,background,3,intensity,147.70005,,-inf,inf,
7,sepd,background,4,intensity,122.26237,,-inf,inf,
8,sepd,background,5,intensity,163.04903,,-inf,inf,
9,sepd,background,6,intensity,124.58762,,-inf,inf,
10,sepd,background,7,intensity,120.30738,,-inf,inf,


#### Run Fitting

In [30]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),25.95
4,📏 Goodness-of-fit (reduced χ²),3.71
5,"📏 R-factor (Rf, %)",8.29
6,"📏 R-factor squared (Rf², %)",4.19
7,"📏 Weighted R-factor (wR, %)",7.04


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4308,5.4309,0.0000,0.00 % ↑
2,sepd,linked_structure,si,scale,,367.9696,379.1657,0.7275,3.04 % ↑
3,sepd,instrument,,d_to_tof_offset,μs,-8.3525,-8.4309,0.0541,0.94 % ↑
4,sepd,background,1,intensity,,213.5506,203.7778,0.4110,4.58 % ↓
5,sepd,background,2,intensity,,117.6167,103.7259,0.4466,11.81 % ↓
6,sepd,background,3,intensity,,147.7001,125.0930,0.8726,15.31 % ↓
7,sepd,background,4,intensity,,122.2624,119.8258,0.9651,1.99 % ↓
8,sepd,background,5,intensity,,163.0490,127.3483,2.7426,21.90 % ↓
9,sepd,background,6,intensity,,124.5876,123.1671,1.6805,1.14 % ↓
10,sepd,background,7,intensity,,120.3074,121.7372,1.7646,1.19 % ↑


#### Display Pattern

In [31]:
project.display.pattern(expt_name='sepd')

In [32]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 3/4

Fix background points.

In [33]:
for point in expt.background:
    point.intensity.free = False

Set more parameters to be refined.

In [34]:
expt.peak.broad_gauss_sigma_0.free = True
expt.peak.broad_gauss_sigma_1.free = True
expt.peak.broad_lorentz_gamma_1.free = True

Show free parameters after selection.

In [35]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43092,0.00004,-inf,inf,Å
2,sepd,linked_structure,si,scale,379.16574,0.72749,-inf,inf,
3,sepd,peak,,broad_lorentz_gamma_1,2.54890,,-inf,inf,μs/Å
4,sepd,peak,,broad_gauss_sigma_0,3.01480,,-inf,inf,μs²
5,sepd,peak,,broad_gauss_sigma_1,33.34510,,-inf,inf,μs/Å
6,sepd,instrument,,d_to_tof_offset,-8.43094,0.05407,-inf,inf,μs


#### Run Fitting

In [36]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),10.85
4,📏 Goodness-of-fit (reduced χ²),3.63
5,"📏 R-factor (Rf, %)",8.33
6,"📏 R-factor squared (Rf², %)",4.27
7,"📏 Weighted R-factor (wR, %)",6.97


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4309,5.4309,0.0000,0.00 % ↓
2,sepd,linked_structure,si,scale,,379.1657,378.7720,0.7565,0.10 % ↓
3,sepd,peak,,broad_lorentz_gamma_1,μs/Å,2.5489,2.2631,0.0734,11.21 % ↓
4,sepd,peak,,broad_gauss_sigma_0,μs²,3.0148,6.1381,0.3670,103.60 % ↑
5,sepd,peak,,broad_gauss_sigma_1,μs/Å,33.3451,32.7007,0.6815,1.93 % ↓
6,sepd,instrument,,d_to_tof_offset,μs,-8.4309,-8.3734,0.0567,0.68 % ↓


#### Display Pattern

In [37]:
project.display.pattern(expt_name='sepd')

In [38]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

### Perform Fit 4/4

Set more parameters to be refined.

In [39]:
structure.atom_sites['Si'].adp_iso.free = True

expt.peak.decay_beta_0.free = True
expt.peak.decay_beta_1.free = True

Show free parameters after selection.

In [40]:
project.display.parameters.free()

Free parameters for both structures (🧩 data blocks) and experiments (🔬 data blocks)


,datablock,category,entry,parameter,value,uncertainty,min,max,units
1,si,cell,,length_a,5.43088,0.00004,-inf,inf,Å
2,si,atom_site,Si,adp_iso,0.50000,,-inf,inf,Å²
3,sepd,linked_structure,si,scale,378.77202,0.75652,-inf,inf,
4,sepd,peak,,decay_beta_0,0.04080,,-inf,inf,μs
5,sepd,peak,,decay_beta_1,0.01230,,-inf,inf,μs/Å
6,sepd,peak,,broad_lorentz_gamma_1,2.26315,0.07343,-inf,inf,μs/Å
7,sepd,peak,,broad_gauss_sigma_0,6.13808,0.36698,-inf,inf,μs²
8,sepd,peak,,broad_gauss_sigma_1,32.70066,0.68151,-inf,inf,μs/Å
9,sepd,instrument,,d_to_tof_offset,-8.37340,0.05668,-inf,inf,μs


#### Run Fitting

In [41]:
project.analysis.fit()
project.display.fit.results()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'sepd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.
2,chi_square_change_tolerance,0.01,Relative change in the objective (chi-square) used to stop fitting.
3,parameter_change_tolerance,1e-08,Relative change in fitted parameters used to stop fitting.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),15.01
4,📏 Goodness-of-fit (reduced χ²),3.60
5,"📏 R-factor (Rf, %)",8.26
6,"📏 R-factor squared (Rf², %)",4.24
7,"📏 Weighted R-factor (wR, %)",6.94


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,cell,,length_a,Å,5.4309,5.4309,0.0001,0.00 % ↓
2,si,atom_site,Si,adp_iso,Å²,0.5000,0.5194,0.0036,3.88 % ↑
3,sepd,linked_structure,si,scale,,378.7720,382.7579,1.0333,1.05 % ↑
4,sepd,peak,,decay_beta_0,μs,0.0408,0.0406,0.0002,0.47 % ↓
5,sepd,peak,,decay_beta_1,μs/Å,0.0123,0.0124,0.0002,0.92 % ↑
6,sepd,peak,,broad_lorentz_gamma_1,μs/Å,2.2631,2.2764,0.0797,0.59 % ↑
7,sepd,peak,,broad_gauss_sigma_0,μs²,6.1381,5.9666,0.4338,2.79 % ↓
8,sepd,peak,,broad_gauss_sigma_1,μs/Å,32.7007,32.2943,0.6972,1.24 % ↓
9,sepd,instrument,,d_to_tof_offset,μs,-8.3734,-8.3777,0.0810,0.05 % ↑


#### Display Correlations

In [42]:
project.display.fit.correlations()

#### Display Pattern

In [43]:
project.display.pattern(expt_name='sepd')

In [44]:
project.display.pattern(expt_name='sepd', x_min=23200, x_max=23700)

In [45]:
project.display.pattern(expt_name='sepd', x='d_spacing')

## 💾 Save Project

In [46]:
project.save_as(dir_path='projects/refine-si-sepd')

Saving project 📦 'si_sepd' to '../../../projects/refine-si-sepd'


├── 📄 project.edi
├── 📁 structures/
│   └── 📄 si.edi
├── 📁 experiments/
│   └── 📄 sepd.edi
├── 📁 analysis/
│   └── 📄 analysis.edi
└── 📁 reports/
    └── 📄 si_sepd.html
